# Appendix 12. Class Imbalance 기초

## 1. Goal

이 notebook은 classification target의 class 비율이 크게 다른 상황을 진단하고 평가·학습·의사결정 단계에서 다루는 방법을 익힙니다. Binary class imbalance를 중심으로 설명하고 마지막에 multiclass로 확장합니다.

학습 목표는 다음과 같습니다.

- class count, prevalence, support와 imbalance ratio를 계산합니다.
- stratified split, baseline과 불균형에 맞는 metric을 사용합니다.
- `class_weight`, random over/under-sampling과 SMOTE의 차이를 설명합니다.
- sampler를 train fold 안에서만 실행해 data leakage를 막습니다.
- validation prevalence를 유지한 상태에서 threshold와 운영 capacity를 검토합니다.

불균형은 data의 특성이지 자동으로 고쳐야 할 오류는 아닙니다. Rare class의 의미, label 품질, sampling process와 실패 비용을 먼저 확인한 뒤 전략을 선택합니다. 모든 결과는 고정 seed의 합성 data를 사용한 API 실습이며 실제 model evidence가 아닙니다.

## 2. Setup

`scikit-learn`으로 data, split, model과 metric을 구성하고 `imbalanced-learn`의 sampler-aware Pipeline으로 resampling을 train boundary 안에 둡니다. Validation data는 실제 평가 population의 prevalence를 보존합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

SEED = 20260714
rng = np.random.default_rng(SEED)

COLORS = {
    "blue": "#2F6B9A",
    "blue_light": "#A9C6DC",
    "gold": "#C6922D",
    "orange": "#D66A3A",
    "ink": "#27313A",
    "gray": "#8A949E",
    "grid": "#E5E9ED",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": COLORS["ink"],
    "axes.labelcolor": COLORS["ink"],
    "axes.titlecolor": COLORS["ink"],
    "axes.grid": True,
    "grid.color": COLORS["grid"],
    "grid.linewidth": 0.8,
    "font.size": 10,
})

pd.set_option("display.max_columns", 20)

## 3. Steps

### 3-1. 불균형의 크기와 원인을 확인하기

#### 3-1-1. count, prevalence와 imbalance ratio

Class imbalance는 class별 sample 수가 크게 다른 상태입니다. Positive prevalence는 전체 sample 중 positive 비율이고, imbalance ratio는 여기서 majority count를 minority count로 나눈 값으로 정의합니다. 보고서에는 비율만 쓰지 말고 실제 support를 함께 적습니다. Positive가 5%여도 20건인지 20,000건인지에 따라 학습과 평가의 불확실성이 다릅니다.

In [ ]:
feature_array, target_array = make_classification(
    n_samples=1_800,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_clusters_per_class=2,
    weights=[0.94, 0.06],
    flip_y=0.015,
    class_sep=1.05,
    random_state=SEED,
)

feature_names = [f"feature_{index}" for index in range(feature_array.shape[1])]
features = pd.DataFrame(feature_array, columns=feature_names)
target = pd.Series(target_array, name="event")

class_profile = target.value_counts().sort_index().rename("count").to_frame()
class_profile["share"] = class_profile["count"] / class_profile["count"].sum()
class_profile.index = ["negative", "positive"]
imbalance_ratio = class_profile["count"].max() / class_profile["count"].min()

print({"samples": len(target), "imbalance_ratio": imbalance_ratio})
class_profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
bar_colors = [COLORS["blue_light"], COLORS["orange"]]

axes[0].bar(class_profile.index, class_profile["count"], color=bar_colors, edgecolor=COLORS["ink"])
axes[0].set(title="Class support", xlabel="class", ylabel="samples", ylim=(0, class_profile["count"].max() * 1.12))
for index, value in enumerate(class_profile["count"]):
    axes[0].text(index, value, f"{value:,}", ha="center", va="bottom")

axes[1].bar(class_profile.index, class_profile["share"], color=bar_colors, edgecolor=COLORS["ink"])
axes[1].set(title="Class prevalence", xlabel="class", ylabel="share", ylim=(0, 1))
for index, value in enumerate(class_profile["share"]):
    axes[1].text(index, value, f"{value:.1%}", ha="center", va="bottom")
plt.show()

#### 3-1-2. 불균형이 생긴 과정을 먼저 조사하기

Rare class가 실제 현상 자체의 낮은 발생률을 반영할 수도 있고, measurement·labeling·sampling 과정에서 누락되어 적게 보일 수도 있습니다. 다음 질문을 먼저 확인합니다.

- Target 정의와 prediction 시점이 올바른가?
- Positive label의 false negative와 adjudication 차이가 큰가?
- 특정 site, 장비, 기간이나 cohort에서 positive가 빠졌는가?
- 한 patient의 여러 row가 count를 부풀리지 않았는가?
- 미래 운영 population의 prevalence와 현재 train data가 같은가?

Class를 50:50으로 만드는 것이 목표가 아닙니다. Model이 사용될 population의 risk와 의사결정 비용을 반영하는 것이 목표입니다.

### 3-2. Split과 baseline을 먼저 고정하기

#### 3-2-1. stratified split로 class 비율 보존하기

Random split에서는 rare class 수가 split마다 크게 흔들릴 수 있습니다. `train_test_split(..., stratify=y)`는 train과 validation의 class 비율을 비슷하게 유지합니다. 다만 같은 patient의 row가 양쪽에 들어가는 문제나 시간 순서를 해결하지는 않습니다. Group 또는 temporal boundary가 필요하면 먼저 그 경계를 지키고, 가능한 범위에서 stratification을 적용합니다.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=SEED,
    stratify=target,
)

split_profile = pd.DataFrame({
    "all": target.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "valid": y_valid.value_counts(normalize=True).sort_index(),
})
split_profile.index = ["negative", "positive"]
split_profile

#### 3-2-2. majority baseline과 plain model

Accuracy는 전체 sample 중 맞춘 비율이므로 majority class가 매우 크면 모든 sample을 negative로 예측해도 높게 나옵니다. `DummyClassifier`를 같은 validation split에서 평가해 이 착시를 확인합니다. Balanced accuracy, positive precision·recall·F1, AUROC, PR-AUC와 AP를 함께 계산합니다.

In [ ]:
def evaluate_binary_predictions(name, y_true, probability, threshold=0.5):
    '''Summarize label and ranking metrics at one threshold.'''
    predicted = (probability >= threshold).astype(int)
    curve_precision, curve_recall, _ = precision_recall_curve(y_true, probability)
    return pd.Series({
        "accuracy": accuracy_score(y_true, predicted),
        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted),
        "f1": f1_score(y_true, predicted, zero_division=0),
        "auroc": roc_auc_score(y_true, probability),
        "pr_auc_trapezoid": auc(curve_recall, curve_precision),
        "average_precision": average_precision_score(y_true, probability),
        "predicted_positive_rate": predicted.mean(),
        "mean_probability": probability.mean(),
    }, name=name)


dummy_model = DummyClassifier(strategy="prior")
dummy_model.fit(X_train, y_train)
dummy_probability = dummy_model.predict_proba(X_valid)[:, 1]

plain_model = SklearnPipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1_000)),
])
plain_model.fit(X_train, y_train)
plain_probability = plain_model.predict_proba(X_valid)[:, 1]

baseline_comparison = pd.concat(
    [
        evaluate_binary_predictions("dummy_prior", y_valid, dummy_probability),
        evaluate_binary_predictions("plain_logistic", y_valid, plain_probability),
    ],
    axis=1,
).T
baseline_comparison

#### 3-2-3. accuracy가 가리는 confusion matrix

Majority baseline의 accuracy가 높아도 positive recall은 0일 수 있습니다. Confusion matrix의 실제 count를 확인하면 false negative가 얼마나 되는지 바로 드러납니다. Metric은 count를 대체하지 않으며, validation support가 작으면 한두 건으로 recall이 크게 바뀔 수 있습니다.

In [ ]:
dummy_predicted = (dummy_probability >= 0.5).astype(int)
plain_predicted = (plain_probability >= 0.5).astype(int)

confusion_comparison = pd.DataFrame(
    {
        "dummy_prior": confusion_matrix(y_valid, dummy_predicted, labels=[0, 1]).ravel(),
        "plain_logistic": confusion_matrix(y_valid, plain_predicted, labels=[0, 1]).ravel(),
    },
    index=["true_negative", "false_positive", "false_negative", "true_positive"],
)
confusion_comparison

### 3-3. 불균형에 맞는 metric 읽기

#### 3-3-1. AUROC와 precision-recall curve

AUROC는 positive와 negative의 ranking을 전체 false-positive-rate 범위에서 평가합니다. Negative가 매우 많을 때 false positive count가 커도 false-positive-rate는 작게 보일 수 있습니다. Precision-recall curve는 positive prediction의 purity와 positive coverage를 직접 보여주며, PR-AUC와 AP의 기준은 positive prevalence에 영향을 받습니다.

이 notebook의 `pr_auc_trapezoid`는 curve 점 사이를 직선으로 연결해 적분합니다. AP는 recall 증가분으로 precision을 가중하는 step 방식입니다. Tied score가 많은 constant baseline에서는 trapezoidal PR-AUC가 낙관적으로 보일 수 있으므로 AP와 같은 값으로 취급하지 않습니다. 어느 하나가 항상 우월한 것은 아니며 계산 방법, prevalence, threshold metric과 실제 count를 함께 기록합니다.

In [ ]:
plain_precision_curve, plain_recall_curve, _ = precision_recall_curve(
    y_valid,
    plain_probability,
)
plain_pr_auc_trapezoid = auc(plain_recall_curve, plain_precision_curve)
plain_average_precision = average_precision_score(y_valid, plain_probability)

fig, ax = plt.subplots(figsize=(7.2, 4.8), layout="constrained")
ax.plot(
    plain_recall_curve,
    plain_precision_curve,
    color=COLORS["blue"],
    linewidth=2,
    label=(
        f"plain logistic "
        f"(trapezoidal PR-AUC={plain_pr_auc_trapezoid:.3f}, "
        f"AP={plain_average_precision:.3f})"
    ),
)
ax.axhline(
    y_valid.mean(),
    color=COLORS["gray"],
    linestyle="--",
    label=f"positive prevalence ({y_valid.mean():.1%})",
)
ax.set(
    title=f"Precision-recall curve (validation n={len(y_valid)})",
    xlabel="recall",
    ylabel="precision",
    xlim=(0, 1),
    ylim=(0, 1),
)
ax.legend(frameon=False)
plt.show()

#### 3-3-2. threshold metric과 ranking metric 구분

AUROC, PR-AUC와 AP는 threshold를 정하기 전 score ranking을 평가합니다. Precision, recall, specificity, F1과 alert rate는 threshold 이후의 decision을 평가합니다. 불균형을 다룰 때 model 학습 전략과 threshold 정책을 한꺼번에 바꾸면 어떤 변경이 효과를 냈는지 알기 어렵습니다. 먼저 같은 threshold와 같은 validation data로 model strategy를 비교하고, 그 다음 threshold를 검토합니다.

### 3-4. Class weight로 학습 objective 조정하기

#### 3-4-1. balanced class weight 계산

`compute_class_weight(class_weight="balanced")`는 class frequency의 역수에 비례하는 weight를 계산합니다. Rare positive error가 학습 objective에 더 크게 반영됩니다. Sample을 복제하거나 삭제하지 않으므로 모든 row는 유지됩니다.

Weight는 label quality를 개선하지 않으며 probability의 calibration이나 운영 threshold를 자동으로 해결하지 않습니다. Weighting policy는 train data에서 계산하고 validation은 원래 분포로 평가합니다.

In [ ]:
class_labels = np.array([0, 1])
balanced_weights = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=y_train,
)
class_weight_table = pd.DataFrame({
    "class": class_labels,
    "train_count": y_train.value_counts().sort_index().to_numpy(),
    "balanced_weight": balanced_weights,
})
class_weight_table

#### 3-4-2. `class_weight="balanced"` model 비교

같은 preprocessing과 estimator를 유지하고 class weight만 바꿉니다. Weighted model은 일반적으로 positive recall과 predicted-positive rate를 높이지만 precision이 낮아질 수 있습니다. 결과는 data와 model에 따라 달라지므로 validation metric과 count로 확인합니다.

In [ ]:
weighted_model = SklearnPipeline([
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(class_weight="balanced", max_iter=1_000),
    ),
])
weighted_model.fit(X_train, y_train)
weighted_probability = weighted_model.predict_proba(X_valid)[:, 1]

weight_comparison = pd.concat(
    [
        evaluate_binary_predictions("plain", y_valid, plain_probability),
        evaluate_binary_predictions("class_weight", y_valid, weighted_probability),
    ],
    axis=1,
).T
weight_comparison

In [ ]:
metrics_to_plot = ["precision", "recall", "balanced_accuracy", "average_precision"]
ax = weight_comparison[metrics_to_plot].T.plot.bar(
    figsize=(8.5, 4.8),
    color=[COLORS["blue"], COLORS["orange"]],
    edgecolor=COLORS["ink"],
    width=0.72,
)
ax.set(
    title=f"Plain and class-weighted model metrics (validation n={len(y_valid)})",
    xlabel="metric",
    ylabel="value",
    ylim=(0, 1),
)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="training strategy", frameon=False, ncol=2)
plt.show()

#### 3-4-3. class weight와 sample weight

Class weight는 같은 class의 sample에 공통 weight를 줍니다. `sample_weight`를 지원하는 estimator와 metric은 row마다 다른 중요도를 줄 수 있습니다. Sampling probability 보정, cohort importance 또는 label confidence처럼 명확한 근거가 있을 때만 사용하고 weight의 출처와 normalization을 contract에 기록합니다. 동일한 현상을 class weight와 sample weight로 중복 보정하지 않습니다.

### 3-5. Train data resampling 비교하기

#### 3-5-1. random over-sampling과 under-sampling

`RandomOverSampler`는 minority sample을 replacement와 함께 다시 뽑습니다. Data를 버리지 않지만 같은 관측이 반복되어 overfitting 가능성이 있습니다. `RandomUnderSampler`는 majority sample을 줄여 학습을 빠르게 할 수 있지만 정보가 사라지고 결과가 random seed에 민감할 수 있습니다.

아래에서는 train data만 resampling하고 validation data는 건드리지 않습니다.

In [ ]:
random_over_sampler = RandomOverSampler(
    sampling_strategy=0.5,
    random_state=SEED,
)
random_under_sampler = RandomUnderSampler(
    sampling_strategy=0.5,
    random_state=SEED,
)

X_over, y_over = random_over_sampler.fit_resample(X_train, y_train)
X_under, y_under = random_under_sampler.fit_resample(X_train, y_train)

sampling_counts = pd.DataFrame({
    "original_train": y_train.value_counts().sort_index(),
    "random_over": pd.Series(y_over).value_counts().sort_index(),
    "random_under": pd.Series(y_under).value_counts().sort_index(),
})
sampling_counts.index = ["negative", "positive"]
sampling_counts

#### 3-5-2. SMOTE의 interpolation과 한계

SMOTE는 minority sample과 이웃 사이를 interpolation해 synthetic sample을 만듭니다. 단순 복제와 달리 새로운 numeric point를 만들지만 다음 조건을 확인해야 합니다.

- Distance가 의미 있도록 numeric feature의 scale을 맞춥니다.
- Outlier나 잘못된 label 주변에서 synthetic sample을 만들 수 있습니다.
- One-hot encoded category 사이를 interpolation하면 현실에 없는 값이 생길 수 있습니다. Mixed feature에는 `SMOTENC`, category-only data에는 `SMOTEN` 같은 variant를 검토합니다.
- Time, patient, site boundary를 무시하고 이웃을 만들면 안 됩니다.
- Synthetic sample은 실제 minority evidence가 늘어난 것이 아닙니다.

여기서는 scale 이후 SMOTE가 실행되도록 Pipeline 순서를 명시합니다.

In [ ]:
scaled_train = StandardScaler().fit_transform(X_train)
smote_preview = SMOTE(sampling_strategy=0.5, random_state=SEED, k_neighbors=5)
X_smote_preview, y_smote_preview = smote_preview.fit_resample(scaled_train, y_train)
sampling_counts["smote"] = pd.Series(y_smote_preview).value_counts().sort_index().to_numpy()

ax = sampling_counts.T.plot.bar(
    figsize=(8.5, 4.8),
    color=[COLORS["blue_light"], COLORS["orange"]],
    edgecolor=COLORS["ink"],
    width=0.72,
)
ax.set(
    title="Train class support after each sampling strategy",
    xlabel="strategy",
    ylabel="samples",
)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="class", frameon=False)
plt.show()

#### 3-5-3. sampler-aware Pipeline로 model 비교

scikit-learn Pipeline은 transformer와 estimator를 연결하지만 sampler의 `fit_resample` contract를 알지 못합니다. `imblearn.pipeline.Pipeline`은 sampler를 train 중에만 실행하고 inference에서는 건너뜁니다. Scale, sampler, model을 하나의 Pipeline으로 묶으면 cross-validation fold마다 올바른 train 부분만 resampling됩니다.

In [ ]:
sampling_models = {
    "plain": plain_model,
    "class_weight": weighted_model,
    "random_over": ImbPipeline([
        ("scale", StandardScaler()),
        ("sample", RandomOverSampler(sampling_strategy=0.5, random_state=SEED)),
        ("model", LogisticRegression(max_iter=1_000)),
    ]),
    "random_under": ImbPipeline([
        ("scale", StandardScaler()),
        ("sample", RandomUnderSampler(sampling_strategy=0.5, random_state=SEED)),
        ("model", LogisticRegression(max_iter=1_000)),
    ]),
    "smote": ImbPipeline([
        ("scale", StandardScaler()),
        ("sample", SMOTE(sampling_strategy=0.5, random_state=SEED, k_neighbors=5)),
        ("model", LogisticRegression(max_iter=1_000)),
    ]),
}

strategy_rows = []
strategy_probabilities = {}
for strategy_name, model in sampling_models.items():
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]
    strategy_probabilities[strategy_name] = probability
    strategy_rows.append(
        evaluate_binary_predictions(strategy_name, y_valid, probability)
    )

strategy_comparison = pd.DataFrame(strategy_rows)
strategy_comparison

In [ ]:
strategy_plot_metrics = ["precision", "recall", "average_precision"]
ax = strategy_comparison[strategy_plot_metrics].plot.bar(
    figsize=(9.5, 5),
    color=[COLORS["blue"], COLORS["orange"], COLORS["gold"]],
    edgecolor=COLORS["ink"],
    width=0.76,
)
ax.set(
    title=f"Validation metrics by imbalance strategy (n={len(y_valid)})",
    xlabel="training strategy",
    ylabel="metric value",
    ylim=(0, 1),
)
ax.tick_params(axis="x", rotation=20)
ax.legend(frameon=False, ncol=3)
plt.show()

#### 3-5-4. resampling과 probability 해석

Resampling과 class weight는 model이 학습 중에 보는 effective class prior를 바꿉니다. Ranking이나 recall이 좋아져도 `predict_proba`의 평균이 validation prevalence와 크게 다를 수 있습니다. Probability를 실제 risk로 사용한다면 original-prevalence validation data에서 calibration을 별도로 검토합니다. Threshold도 resampled train prevalence가 아니라 운영 비용과 capacity로 정합니다.

In [ ]:
probability_profile = strategy_comparison[
    ["mean_probability", "predicted_positive_rate", "auroc", "average_precision"]
].copy()
probability_profile.insert(0, "validation_prevalence", y_valid.mean())
probability_profile

### 3-6. Resampling leakage와 cross-validation

#### 3-6-1. split 전에 over-sampling하지 않기

전체 data를 먼저 over-sampling한 뒤 split하면 복제된 minority row가 train과 validation 양쪽에 들어갈 수 있습니다. Validation이 더 이상 unseen sample을 나타내지 않으므로 성능이 낙관적으로 보입니다. 아래에서는 row id만 resampling해 실제로 몇 개의 원본 id가 양쪽에 겹치는지 확인합니다.

In [ ]:
row_ids = np.arange(len(target)).reshape(-1, 1)
resampled_ids, resampled_target = RandomOverSampler(
    sampling_strategy=0.5,
    random_state=SEED,
).fit_resample(row_ids, target)

bad_train_ids, bad_valid_ids = train_test_split(
    resampled_ids.ravel(),
    test_size=0.25,
    random_state=SEED,
    stratify=resampled_target,
)
leaked_original_ids = np.intersect1d(bad_train_ids, bad_valid_ids)

print({
    "resampled_rows": len(resampled_ids),
    "original_ids_present_in_both_splits": len(leaked_original_ids),
})

#### 3-6-2. StratifiedKFold 안에서 sampler 실행하기

Cross-validation에서도 sampler는 각 fold의 train 부분에서만 fit되어야 합니다. `ImbPipeline` 전체를 `cross_validate`에 넘기면 이 경계를 지킬 수 있습니다. `StratifiedKFold`는 각 fold의 class 비율을 비슷하게 유지하지만 group·time leakage를 해결하지 않으므로 실제 grain에 맞는 splitter가 우선입니다.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scoring = {
    "auroc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
}

cv_rows = []
for strategy_name, model in sampling_models.items():
    cv_result = cross_validate(
        model,
        features,
        target,
        cv=cv,
        scoring=cv_scoring,
        n_jobs=1,
    )
    cv_rows.append({
        "strategy": strategy_name,
        "auroc_mean": cv_result["test_auroc"].mean(),
        "auroc_std": cv_result["test_auroc"].std(ddof=1),
        "ap_mean": cv_result["test_average_precision"].mean(),
        "ap_std": cv_result["test_average_precision"].std(ddof=1),
        "balanced_accuracy_mean": cv_result["test_balanced_accuracy"].mean(),
    })

cv_comparison = pd.DataFrame(cv_rows).set_index("strategy")
cv_comparison

### 3-7. Threshold와 운영 capacity 연결하기

#### 3-7-1. precision, recall과 alert rate 비교

불균형 문제의 최종 목적은 class 수를 같게 만드는 것이 아니라 제한된 capacity에서 중요한 positive를 찾는 것입니다. Threshold별 precision, recall, false-positive count와 alert rate를 계산합니다. Threshold는 validation에서 policy로 정하고 sealed test에 맞춰 조정하지 않습니다.

In [ ]:
threshold_probability = strategy_probabilities["class_weight"]
threshold_rows = []
for threshold in np.linspace(0.10, 0.90, 17):
    predicted = (threshold_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, predicted, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_valid, predicted, zero_division=0),
        "recall": recall_score(y_valid, predicted),
        "alert_rate": predicted.mean(),
        "false_positive_count": fp,
        "true_positive_count": tp,
    })

threshold_table = pd.DataFrame(threshold_rows)
threshold_table.loc[threshold_table["recall"].ge(0.70)].tail(5)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8), layout="constrained")
for column, color, linestyle in [
    ("precision", COLORS["blue"], "-"),
    ("recall", COLORS["orange"], "-"),
    ("alert_rate", COLORS["gray"], "--"),
]:
    ax.plot(
        threshold_table["threshold"],
        threshold_table[column],
        color=color,
        linestyle=linestyle,
        linewidth=2,
        label=column,
    )

ax.set(
    title=f"Class-weighted model by threshold (validation n={len(y_valid)})",
    xlabel="threshold",
    ylabel="metric value / prediction share",
    xlim=(0.1, 0.9),
    ylim=(0, 1),
)
ax.legend(frameon=False, ncol=3)
plt.show()

### 3-8. Multiclass imbalance로 확장하기

#### 3-8-1. class별 support와 weight

Multiclass에서는 가장 작은 class 하나만 보지 말고 모든 class의 support를 확인합니다. `compute_class_weight`는 class별 weight를 만들고, macro metric은 각 class에 같은 비중을 둡니다. Weighted average는 support가 큰 class의 성능을 더 크게 반영하므로 rare class 실패를 숨길 수 있습니다.

In [ ]:
multiclass_target = pd.Series(
    rng.choice(
        ["low", "moderate", "severe"],
        size=900,
        p=[0.74, 0.21, 0.05],
    ),
    name="severity",
)
multiclass_labels = np.array(["low", "moderate", "severe"])
multiclass_weights = compute_class_weight(
    class_weight="balanced",
    classes=multiclass_labels,
    y=multiclass_target,
)

multiclass_profile = pd.DataFrame({
    "support": multiclass_target.value_counts().reindex(multiclass_labels),
    "balanced_weight": multiclass_weights,
})
multiclass_profile

#### 3-8-2. multiclass `sampling_strategy`

Multiclass sampler의 `sampling_strategy`를 dict로 주면 목표 class별 resampled count를 명시할 수 있습니다. 모든 class를 majority count까지 늘리는 대신 충분한 이유가 있는 목표 count를 정합니다. SMOTE와 ADASYN 계열은 각 대상 class를 one-vs-rest 방식으로 다루지만 class overlap, rare support와 neighbor 수를 점검해야 합니다.

In [ ]:
multiclass_features = pd.DataFrame(
    rng.normal(size=(len(multiclass_target), 3)),
    columns=["signal_1", "signal_2", "signal_3"],
)
multiclass_sampler = RandomOverSampler(
    sampling_strategy={"moderate": 300, "severe": 200},
    random_state=SEED,
)
_, multiclass_resampled_target = multiclass_sampler.fit_resample(
    multiclass_features,
    multiclass_target,
)

multiclass_sampling_counts = pd.DataFrame({
    "original": multiclass_target.value_counts().reindex(multiclass_labels),
    "resampled": pd.Series(multiclass_resampled_target).value_counts().reindex(multiclass_labels),
})
multiclass_sampling_counts

### 3-9. 전략 선택과 monitoring contract

#### 3-9-1. 전략별 trade-off 정리

| 전략 | 장점 | 주요 위험 |
|---|---|---|
| metric·threshold 조정 | data를 바꾸지 않고 의사결정 비용 반영 | 학습 자체의 minority 무시를 해결하지 못할 수 있음 |
| `class_weight` | 모든 row 유지, 적용이 단순함 | probability prior와 threshold를 다시 검토해야 함 |
| random over-sampling | minority 정보를 버리지 않음 | 중복과 overfitting 가능성 |
| random under-sampling | 빠르고 단순함 | majority 정보 손실과 seed 민감성 |
| SMOTE 계열 | minority space에 synthetic point 생성 | scale, outlier, category, boundary에 민감함 |

전략은 cross-validation의 primary metric, guardrail, calibration, class/slice count와 운영 capacity를 함께 비교해 선택합니다.

#### 3-9-2. production에서 prevalence drift 감시

Training이 끝난 뒤에도 target prevalence, predicted-positive rate, score distribution과 class별 label delay를 monitoring합니다. Prevalence가 달라지면 AP·precision과 calibration도 달라질 수 있습니다. Label이 늦게 도착하는 환경에서는 prediction monitoring과 delayed outcome evaluation을 분리합니다. Threshold나 sampling policy 변경은 versioned configuration과 새 validation evidence로 관리합니다.

## 4. Checks

Class support, split prevalence, resampling boundary와 metric 범위를 확인합니다. 이 check는 실제 data의 label audit와 release gate를 대신하지 않습니다.

In [ ]:
assert target.mean() < 0.10
assert imbalance_ratio > 10
assert abs(y_train.mean() - y_valid.mean()) < 0.01
assert baseline_comparison.loc["dummy_prior", "accuracy"] > 0.90
assert baseline_comparison.loc["dummy_prior", "recall"] == 0
assert baseline_comparison.loc["plain_logistic", "average_precision"] > y_valid.mean()

assert balanced_weights[1] > balanced_weights[0]
assert pd.Series(y_over).value_counts().loc[1] > y_train.value_counts().loc[1]
assert pd.Series(y_under).value_counts().loc[0] < y_train.value_counts().loc[0]
assert pd.Series(y_smote_preview).value_counts().loc[1] > y_train.value_counts().loc[1]
assert len(leaked_original_ids) > 0

assert set(strategy_comparison.index) == set(sampling_models)
assert strategy_comparison[["auroc", "average_precision"]].apply(
    lambda column: column.between(0, 1).all()
).all()
assert threshold_table["alert_rate"].between(0, 1).all()
assert multiclass_sampling_counts.loc["severe", "resampled"] == 200

print("All appendix checks passed.")

## 5. Next Steps

- 실제 data에서는 patient·encounter·time boundary를 지키는 split을 먼저 설계합니다.
- Positive label quality와 cohort별 missingness를 audit한 뒤 imbalance strategy를 비교합니다.
- AUROC, PR-AUC·AP, threshold metric, confusion count와 calibration을 함께 기록합니다.
- Sampler는 Pipeline 안에서 train fold에만 적용하고 validation·test prevalence는 보존합니다.
- Regression의 rare tail, multilabel imbalance와 anomaly detection은 target 구조가 다르므로 별도 evaluation contract로 다룹니다.
- 다음 MLflow notebook에서는 strategy, sampling ratio, threshold와 validation evidence를 Run에 기록합니다.

## 6. References

- [scikit-learn: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn API: `train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [scikit-learn API: `compute_class_weight`](https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html)
- [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- [imbalanced-learn: Over-sampling](https://imbalanced-learn.org/stable/over_sampling.html)
- [imbalanced-learn: Under-sampling](https://imbalanced-learn.org/stable/under_sampling.html)
- [imbalanced-learn: Common pitfalls and data leakage](https://imbalanced-learn.org/stable/common_pitfalls.html)
- [imbalanced-learn API: Pipeline](https://imbalanced-learn.org/stable/references/generated/imblearn.pipeline.Pipeline.html)